# Intro Workshop: RAG with LlamaIndex + Ollama (Local)

This notebook is for running the workshop locally (for example VS Code Jupyter).
You will follow a clear progression: prepare a source file, set up Ollama, compare no-RAG vs RAG, then add reranking and routing.

## Learning Goals

By the end, you will be able to:
1. Explain why RAG helps with LLM knowledge cutoffs
2. Build a simple RAG pipeline with LlamaIndex and Ollama
3. Improve retrieval quality with reranking
4. Route queries between a fact tool and a summary tool

### Section Map

- Section 1: Setup Ollama locally
- Section 2: Load local source data
- Section 3: Core learning loop (no-RAG failure -> RAG success)
- Section 4: Advanced retrieval with reranking
- Section 5: Intent routing with two query tools

If you restart your kernel or Ollama service, rerun cells in order.

## 1) Setup Ollama Locally

### What This Section Does

In this section you will:
- Install required Python packages into your active environment
- Verify the local Ollama CLI is available
- Start Ollama (if needed) and pull models
- Run a quick smoke test

### 1.1 Install Python Dependencies

Install core LlamaIndex + Ollama packages for your local environment.

In [ ]:
# Local setup: install Python dependencies
%pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama requests beautifulsoup4

### 1.2 Verify Ollama Installation

Ensure the `ollama` CLI is installed and available on PATH.

In [ ]:
# Local setup: verify ollama CLI is available
import shutil

OLLAMA_CMD = shutil.which("ollama") or "ollama"
if shutil.which("ollama") is None:
    raise RuntimeError(
        "Ollama CLI not found. Install from https://ollama.com/download and restart your terminal/kernel."
    )

print(f"Using Ollama CLI: {OLLAMA_CMD}")

### 1.3 Start The Ollama Service

Start Ollama at `127.0.0.1:11434` if it is not already running.

In [ ]:
# Local setup: start ollama server if needed
import os
import subprocess
import time
import urllib.request

OLLAMA_CMD = globals().get("OLLAMA_CMD", "ollama")
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

def ollama_api_up(url="http://127.0.0.1:11434/api/tags") -> bool:
    try:
        urllib.request.urlopen(url, timeout=2).close()
        return True
    except Exception:
        return False

if not ollama_api_up():
    subprocess.Popen([OLLAMA_CMD, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(12):
        if ollama_api_up():
            break
        time.sleep(1)

if not ollama_api_up():
    raise RuntimeError("Ollama server is not reachable at http://127.0.0.1:11434")

print("Ollama server is running.")
subprocess.run([OLLAMA_CMD, "list"], check=False)

### 1.4 Pull Models (First Run Only)

We pull two models:
- `llama3.2:3b` for generation
- `nomic-embed-text` for embeddings

Local note: once pulled, models are persisted on your machine and usually do not need re-pulling.

In [ ]:
# Local setup: pull required models
import subprocess

LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

for model_name in [LLM_MODEL, EMBED_MODEL]:
    print(f"Pulling {model_name}...")
    subprocess.run([OLLAMA_CMD, "pull", model_name], check=True)

print(f"Using LLM model: {LLM_MODEL}")
print(f"Using embedding model: {EMBED_MODEL}")

### 1.5 Quick Smoke Test

In [ ]:
# Local setup: quick smoke test
import subprocess

test = subprocess.run(
    [OLLAMA_CMD, "run", LLM_MODEL, "Hello! Anyone home?"],
    capture_output=True,
    text=True,
    check=True,
)
print(test.stdout.strip())

### Setup Checkpoint

If the smoke test responded correctly, your setup is ready.

If not:
- Re-run Section 1.3 (start service)
- Re-run Section 1.4 (model pull)
- Run `ollama list` in a terminal to verify models are present

## 2) Load Local Source Data

Use this section to validate that the local workshop source file is available.

In [ ]:
# Source prep (Local): validate and preview local source text
from pathlib import Path

source_file = Path("data/new_species_2024.txt")
if not source_file.exists():
    raise FileNotFoundError(
        "Missing data/new_species_2024.txt. Place the source file in the data/ folder and rerun this cell."
    )

final_text = source_file.read_text(encoding="utf-8").strip()

if not final_text:
    raise ValueError("Local source file is empty.")

print(f"Loaded {source_file} ({len(final_text)} chars)")
print(final_text[:500] + "...")

## 3) Intro to RAG

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: answer using those chunks as grounded context

In this section you will see a direct comparison:
- First ask the LLM with no retrieval
- Then ask the same question after indexing the source text

### 3.1 Configure LLM And Embedding Model

We configure two Ollama-backed components:
- an LLM for answer generation
- an embedding model for semantic retrieval

This cell works locally once Section 1 is complete.

In [ ]:
# Configure LlamaIndex to use local Ollama models
import logging
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Quiet noisy HTTP client logs in notebook output
for noisy_logger in ("httpx", "httpcore"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

Settings.llm = Ollama(model=LLM_MODEL, request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)
Settings.chunk_size = 128
Settings.chunk_overlap = 50

print(f"LLM configured: {LLM_MODEL}")
print(f"Embedding model configured: {EMBED_MODEL}")

### 3.2 The Problem: LLM Knowledge Cutoffs

First, ask the local model a recent 2024 question with no retrieval context.
This establishes the failure mode that RAG solves.

In [ ]:
# The same question asked without retrieval context
prompt = "Name me a vegetarian piranha discovered in 2024 and tell me who it was named after."
response = Settings.llm.complete(prompt)

print(f"Query: {prompt}\n")
print("=== Direct LLM response (no RAG) ===")
print(getattr(response, "text", str(response)))

### 3.3 How Basic RAG Works

To build basic RAG, we follow four steps:
1. Chunking: split long text into smaller passages.
2. Embedding: convert each chunk into a vector representation of meaning.
3. Retrieval: embed the query and find the most similar chunks.
4. Generation: answer with the query plus retrieved chunks as context.

This is why the same question can perform better after indexing.

In [ ]:
# Build vector index and rerun the same prompt with retrieval
from pathlib import Path
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

source_file = Path("data/new_species_2024.txt")
if not source_file.exists():
    raise FileNotFoundError(
        "Missing data/new_species_2024.txt. Run Section 2 download cell first, "
        "or generate it via scripts/scrape_nhm_new_species.py."
    )

documents = SimpleDirectoryReader(input_files=[str(source_file)]).load_data()
print(f"Loaded {len(documents)} document(s) from {source_file}.")

vector_index = VectorStoreIndex.from_documents(documents)
vector_query_engine = vector_index.as_query_engine(similarity_top_k=3)

# Short aliases used in later sections
index = vector_index
query_engine = vector_query_engine

In [ ]:
# Generate response with RAG
rag_response = vector_query_engine.query(prompt)
print("=== RAG response ===")
print(rag_response)

### 3.4 Peek Under The Hood

Inspect the retrieved chunks that grounded the RAG answer.

In [ ]:
# Inspect which chunks were retrieved for the RAG answer
for i, node in enumerate(rag_response.source_nodes, 1):
    print(f"--- Retrieved Chunk {i} (Score: {node.score:.4f}) ---")
    print(f"{node.text[:250]}...\n")

### 3.5 Run Diverse RAG Questions

These example queries cover multiple 2024 species discoveries so you can better demonstrate retrieval breadth, not just one fact.

In [ ]:
# Intro RAG: run a diverse new-species question set
sample_questions = [
    "Which species in the NHM 2024 roundup were named after famous figures, and where were they found?",
    "What made the Carmenta brachyclados discovery in South Wales so unusual?",
    "Summarize the range of organism groups in the 2024 roundup (for example moths, beetles, copepods, amphipods) and why this diversity matters.",
    "How does the article connect new-species discovery to conservation and infrastructure decisions?",
]

for i, q in enumerate(sample_questions, 1):
    print(f"\n=== Question {i} ===")
    print(q)
    response = query_engine.query(q)
    print(response)

## 4) Advanced Retrieval

### Why Basic RAG Is Not Always Enough

Vector retrieval can return chunks that share words but are only loosely relevant.
If too much weak context is passed to the LLM, answer quality can drop.

Reranking helps by scoring the retrieved chunks for relevance and keeping only the strongest evidence before synthesis.

### 4.1 Baseline Retrieval

Start with standard vector retrieval so you have a clear baseline before adding reranking.

In [ ]:
# Local baseline retrieval (no reranker)
query = "Which species in the NHM 2024 roundup were named after famous figures, and where were they found?"
baseline_engine = index.as_query_engine(similarity_top_k=8)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

### 4.2 Add LLM Reranking

Reranking scores initially retrieved chunks and keeps only the strongest evidence before synthesis.

In [ ]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

RERANK_TOP_N = 3
reranker = LLMRerank(choice_batch_size=4, top_n=RERANK_TOP_N)
rerank_engine = index.as_query_engine(
    similarity_top_k=12,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)
print(f"Reranked nodes returned: {len(rerank_response.source_nodes)} (top_n={RERANK_TOP_N})")

In [ ]:
# Compare retrieved source chunks
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

In this section, three reranked nodes are expected because `top_n=3` is set for teaching clarity.

## 5) Intent Routing: Your First AI Agent

Fact questions and summary questions often need different retrieval strategies.
Routing lets the LLM choose which tool to use based on the query intent.

This is a foundational agent pattern: tool selection by an LLM policy.
Here we use `RouterQueryEngine` with two tools:
- Vector query engine for specific facts
- Summary query engine for broad overviews

In [ ]:
# Native LlamaIndex routing with tool selection
from llama_index.core import SummaryIndex
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex.from_documents(documents)
summary_query_engine = summary_index.as_query_engine()

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for retrieving specific facts, names, or details about individual species discovered in 2024.",
)

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description="Useful for summarizing the full source, identifying high-level themes, and broad overviews.",
)

router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[summary_tool, vector_tool],
)

print("Router engine ready.")

### 5.1 Test The Router

Run both a summary intent and a specific fact intent, then inspect which tool the router selected.

In [ ]:
# Test routing on two different intents
router_queries = [
    "Summarize the main themes of the 2024 new-species discoveries.",
    "What is Myloplus sauron named after?",
]

for q in router_queries:
    print(f"\n--- Query: {q} ---")
    response = router_engine.query(q)
    print("Router selection:", response.metadata.get("selector_result"))
    print("Final answer:")
    print(response)

### 5.2 Try Your Own Routed Prompt

Ask anything about the 2024 discoveries and inspect how the router decides.

In [ ]:
# Interactive router demo
custom_query = input("Ask a species question or ask for a summary: ").strip()
if custom_query:
    response = router_engine.query(custom_query)
    print("Router selection:", response.metadata.get("selector_result"))
    print("Final answer:")
    print(response)
else:
    print("No query entered.")

### Suggested Exercises

Try these quick experiments:
- Change `similarity_top_k` from 4 to 8 in intro RAG
- Change reranker `top_n` from 5 to 3 or 7
- Change `Settings.chunk_size` from 128 to 250
- Change `Settings.chunk_overlap` from 50 to whatever you like
- Ask multi-part questions and observe whether reranking helps